![Redis](https://redis.io/wp-content/uploads/2024/04/Logotype.svg?auto=webp&quality=85,75&width=120)
# Google ADK Agent with RedisVL MCP and Vertex AI

<a href="https://colab.research.google.com/github/redis-developer/redis-ai-resources/blob/main/python-recipes/MCP/00_google_adk_redisvl_mcp_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Introduction

This notebook shows how to build a [Google ADK](https://google.github.io/adk-docs/) agent that connects to Redis through the [RedisVL MCP](https://docs.redisvl.com/en/stable/user_guide/how_to_guides/mcp.html) server over stdio. We will bootstrap a small movie catalog in Redis, generate an MCP config, wire ADK to the server, and run grounded movie-recommendation prompts.

### Key Concepts

Before diving into code, here is a quick orientation on the three technologies this recipe combines:

**[Model Context Protocol (MCP)](https://modelcontextprotocol.io/)**

MCP is an open standard that lets AI agents discover and call external tools through a uniform interface. An MCP *server* advertises a set of typed tools (search, upsert, etc.), and an MCP *client* inside the agent framework calls them. Communication happens over a *transport* -- in this notebook we use **stdio**, which means the agent framework launches the MCP server as a subprocess and talks to it over stdin/stdout. This keeps everything local and requires no network ports.

**[Google Agent Development Kit (ADK)](https://google.github.io/adk-docs/)**

ADK is Google's open-source framework for building AI agents in Python. It provides `LlmAgent` for wrapping a model with tools, `Runner` for executing multi-turn conversations, and `McpToolset` for connecting to MCP servers. ADK handles the MCP client lifecycle automatically -- it starts the subprocess, discovers the available tools, and maps them into the agent's tool list.

**[RedisVL MCP Server](https://docs.redisvl.com/en/stable/user_guide/how_to_guides/mcp.html)**

RedisVL ships an MCP server (`rvl mcp`) that exposes Redis search indexes as MCP tools. You provide a YAML config that binds an index to the server, and it advertises tools such as `search-records` and `upsert-records`. This gives any MCP-compatible agent framework (ADK, Claude Desktop, Cursor, etc.) direct access to Redis-backed retrieval without custom glue code.


### Architecture

The data flow in this recipe is:

```
User prompt
    |
    v
Google ADK Agent (Vertex AI model)
    |  calls MCP tool via stdio
    v
RedisVL MCP Server (subprocess)
    |  executes hybrid search
    v
Redis (vector + full-text index)
    |  returns ranked results
    v
RedisVL MCP Server
    |  returns structured results
    v
Google ADK Agent
    |  grounds answer in evidence
    v
Final response to user
```

The agent never talks to Redis directly. All data access goes through the MCP tool interface, which means the same Redis index could be shared with other MCP-compatible clients without any code changes.


## What We'll Build

This tutorial is for readers who already know the basics of Redis and Python and want to see a minimal MCP-powered agent pattern end to end.

**By the end you will be able to:**
- load a small JSON movie dataset into a Redis index with RedisVL
- configure RedisVL MCP to expose that index over stdio in read-only mode
- connect a Google ADK `LlmAgent` to the MCP server while using Vertex AI for the model
- inspect the structured Redis-backed evidence the agent retrieved before it answered

## Outline

1. Install packages.
2. Configure Vertex AI and Redis.
3. Load the movie sample data and create a Redis index.
4. Generate an MCP YAML config for RedisVL.
5. Configure the MCP toolset and build the ADK agent.
6. Validate the MCP connection and define interaction helpers.
7. Run a first grounded prompt.
8. Try more prompts.
9. Exercise.


## 1. Install Packages

We will use:
- `redisvl[mcp,sentence-transformers,vertexai]` for indexing, local embeddings, and the MCP server
- `google-adk` for the agent runtime
- `google-cloud-aiplatform` so we can initialize and authenticate Vertex AI cleanly from the notebook


In [ ]:
%pip install -q "redisvl[mcp,sentence-transformers,vertexai]>=0.17.1" "google-adk>=1.0.0" pandas nest_asyncio


### Optional: Install Redis Stack in Colab

If you already have a Redis deployment with Search enabled, you can skip this step and use your own connection details instead.


In [ ]:
# NBVAL_SKIP
!git clone https://github.com/redis-developer/redis-ai-resources.git temp_repo
!mv temp_repo/python-recipes/MCP/resources .
!rm -rf temp_repo

In [ ]:
# NBVAL_SKIP
%%sh
sudo apt-get install lsb-release curl gpg
curl -fsSL https://packages.redis.io/gpg | sudo gpg --dearmor -o /usr/share/keyrings/redis-archive-keyring.gpg
sudo chmod 644 /usr/share/keyrings/redis-archive-keyring.gpg
echo "deb [signed-by=/usr/share/keyrings/redis-archive-keyring.gpg] https://packages.redis.io/deb $(lsb_release -cs) main" | sudo tee /etc/apt/sources.list.d/redis.list
sudo apt-get update
sudo apt-get install redis

redis-server --version
redis-server --daemonize yes --loadmodule /usr/lib/redis/modules/redisearch.so

## 2. Configure Vertex AI and Redis

This notebook uses **Vertex AI only for the agent model**. Redis indexing and RedisVL MCP query embeddings stay local with `HFTextVectorizer`, which keeps the retrieval setup simple and reproducible.


In [ ]:
# NBVAL_SKIP
import os
import sys
import json
import uuid
import warnings
from getpass import getpass
from pathlib import Path

import nest_asyncio
import pandas as pd
import yaml
from IPython.display import display

warnings.filterwarnings("ignore")
nest_asyncio.apply()

IN_COLAB = "google.colab" in sys.modules

PROJECT_ID = os.getenv("GCP_PROJECT_ID")
if not PROJECT_ID:
    PROJECT_ID = getpass("GCP_PROJECT_ID: ")

REGION = os.getenv("GCP_REGION")
if not REGION:
    REGION = input("GCP_REGION [us-central1]: ").strip() or "us-central1"

os.environ["GCP_PROJECT_ID"] = PROJECT_ID
os.environ["GCP_REGION"] = REGION
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = REGION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

REDIS_URL = os.getenv("REDIS_URL")
if not REDIS_URL:
    REDIS_HOST = os.getenv("REDIS_HOST", "localhost")
    REDIS_PORT = os.getenv("REDIS_PORT", "6379")
    REDIS_PASSWORD = os.getenv("REDIS_PASSWORD", "")
    auth_part = f":{REDIS_PASSWORD}@" if REDIS_PASSWORD else ""
    REDIS_URL = f"redis://{auth_part}{REDIS_HOST}:{REDIS_PORT}"

print({
    "project_id": PROJECT_ID,
    "region": REGION,
    "redis_url": REDIS_URL,
    "in_colab": IN_COLAB,
})


In [ ]:
# NBVAL_SKIP
if IN_COLAB:
    from google.colab import auth

    auth.authenticate_user()
    print("Authenticated with Colab credentials.")
else:
    print("If you are running locally, make sure Application Default Credentials are available.")
    print("For example: gcloud auth application-default login")


In [ ]:
import vertexai
from redis import Redis
from redisvl.index import SearchIndex
from redisvl.query import AggregateHybridQuery
from redisvl.utils.vectorize import HFTextVectorizer

vertexai.init(project=PROJECT_ID, location=REGION)

redis_client = Redis.from_url(REDIS_URL)
redis_client.ping()


## 3. Load the Sample Movie Data

For this tutorial we use a small movie catalog with titles, genres, ratings, and short plot descriptions. It is compact enough to inspect by eye, but still rich enough to show how Redis-backed retrieval changes an agent's answers.


In [ ]:
movies_path = Path("resources") / "movies.json"
with open(movies_path, "r", encoding="utf-8") as f:
    movies = json.load(f)


In [ ]:
movies_df = pd.DataFrame(movies)
print(f"Loaded {len(movies_df)} movie entries")
display(movies_df.head())


### Note on the `id` Field

RedisVL MCP reserves `id` for its own response envelope, so we rename the dataset's `id` field to `movie_id` before loading records into Redis. That small change keeps the schema compatible with the MCP server while preserving the original identifier.


In [ ]:
INDEX_NAME = "adk-movies"
INDEX_PREFIX = "movie"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

vectorizer = HFTextVectorizer(model=EMBEDDING_MODEL)

schema = {
    "index": {
        "name": INDEX_NAME,
        "prefix": INDEX_PREFIX,
        "storage_type": "json",
    },
    "fields": [
        {"name": "movie_id", "type": "tag"},
        {"name": "title", "type": "text"},
        {"name": "genre", "type": "tag"},
        {"name": "rating", "type": "numeric"},
        {"name": "description", "type": "text"},
        {
            "name": "embedding",
            "type": "vector",
            "attrs": {
                "algorithm": "flat",
                "dims": vectorizer.dims,
                "distance_metric": "cosine",
                "datatype": "float32",
            },
        },
    ],
}

index = SearchIndex.from_dict(schema, redis_url=REDIS_URL)
index.create(overwrite=True, drop=True)

records = []
for movie in movies:
    record = {
        "movie_id": movie["id"],
        "title": movie["title"],
        "genre": movie["genre"],
        "rating": movie["rating"],
        "description": movie["description"],
    }
    record["embedding"] = vectorizer.embed(record["description"])
    records.append(record)

loaded_keys = index.load(records)
print(f"Loaded {len(loaded_keys)} movie records into Redis index '{INDEX_NAME}'.")
display(pd.DataFrame(records)[["movie_id", "title", "genre", "rating"]].head())


### Sanity-Check the Redis Index

Before adding MCP and ADK, it helps to confirm that hybrid search over the Redis index returns sensible matches.


In [ ]:
user_query = "revenge-driven action story"
embedded_query = vectorizer.embed(user_query)

hybrid_query = AggregateHybridQuery(
    text=user_query,
    text_field_name="description",
    vector=embedded_query,
    vector_field_name="embedding",
    num_results=3,
    return_fields=["movie_id", "title", "genre", "rating", "description"],
)

sanity_results = index.query(hybrid_query)
sanity_df = pd.DataFrame(sanity_results)
display(sanity_df[["title", "genre", "rating", "hybrid_score"]])


## 4. Generate a RedisVL MCP Config

The [MCP config](https://docs.redisvl.com/en/stable/user_guide/how_to_guides/mcp.html) binds one logical server to our existing Redis index. We expose only search in this tutorial by launching the server in read-only mode.

The config has three main sections:
- **`server`** -- connection details (the `redis_url` to connect to).
- **`indexes.<name>.search`** -- controls how the MCP `search-records` tool queries Redis:
  - `type: hybrid` means every search combines full-text (BM25) and vector similarity.
  - `text_scorer: BM25STD` selects the standard BM25 scoring algorithm for the text component.
  - `combination_method: LINEAR` blends the text and vector scores as a weighted sum.
  - `linear_text_weight: 0.3` gives 30% weight to text and 70% to vector similarity.
- **`indexes.<name>.runtime`** -- field mappings and guardrails:
  - `text_field_name` / `vector_field_name` tell the server which schema fields to use for each search modality.
  - `default_embed_text_field` is the field whose content gets embedded when the agent sends a text query.
  - `default_limit` / `max_limit` / `max_result_window` cap result sizes to prevent runaway queries.


In [ ]:
from redisvl.mcp import load_mcp_config

Path("tmp").mkdir(exist_ok=True)
mcp_config_path = Path("tmp/google_adk_movies_mcp.yaml")

mcp_config = {
    "server": {
        "redis_url": REDIS_URL,
    },
    "indexes": {
        "movies": {
            "redis_name": INDEX_NAME,
            "vectorizer": {
                "class": "HFTextVectorizer",
                "model": EMBEDDING_MODEL,
            },
            "search": {
                "type": "hybrid",
                "params": {
                    "text_scorer": "BM25STD",
                    "stopwords": "english",
                    "vector_search_method": "KNN",
                    "combination_method": "LINEAR",
                    "linear_text_weight": 0.3,
                },
            },
            "runtime": {
                "text_field_name": "description",
                "vector_field_name": "embedding",
                "default_embed_text_field": "description",
                "default_limit": 5,
                "max_limit": 10,
                "max_result_window": 100,
            },
        }
    },
}

mcp_config_path.write_text(yaml.safe_dump(mcp_config, sort_keys=False), encoding="utf-8")
validated_config = load_mcp_config(str(mcp_config_path))

print(f"Wrote MCP config to {mcp_config_path.resolve()}")
print({
    "binding_id": validated_config.binding_id,
    "redis_name": validated_config.redis_name,
    "search_type": validated_config.binding.search.type,
})
print(mcp_config_path.read_text(encoding="utf-8"))


## 5. Configure the MCP Toolset and Build the ADK Agent

This section sets up the Google ADK side. We will:
1. Create a `McpToolset` that launches the RedisVL MCP server as a subprocess over stdio.
2. Define an `LlmAgent` that uses the toolset for retrieval and Vertex AI for generation.
3. Set up a `Runner` to manage multi-turn conversation state.

> **Troubleshooting:** The `uvx` command comes from the [uv](https://docs.astral.sh/uv/) package manager. If it is not on your PATH, install it with `pip install uv` or see the [uv installation docs](https://docs.astral.sh/uv/getting-started/installation/). If Vertex AI returns an authentication error, make sure Application Default Credentials are configured (`gcloud auth application-default login`).

### 5a. Import ADK Components and Define the MCP Toolset

We use [`McpToolset`](https://google.github.io/adk-docs/tools/mcp-tools/) with `StdioConnectionParams` to launch the RedisVL MCP server via `uvx --from redisvl[mcp] rvl mcp --config <path> --read-only`. The `tool_filter` restricts which MCP tools the agent can call -- here we only expose `search-records` since this is a read-only recipe.


In [ ]:
from mcp import StdioServerParameters
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from google.genai import types

APP_NAME = "redisvl_mcp_movies_app"
USER_ID = "notebook_user"

# Using Gemini 2.0 Flash -- a fast, cost-effective model that supports tool calling.
# See https://ai.google.dev/gemini-api/docs/models for available model names.
MODEL_NAME = "gemini-2.0-flash"

mcp_server_command = "uvx"
mcp_server_args = [
    "--from",
    "redisvl[mcp]",
    "rvl",
    "mcp",
    "--config",
    str(mcp_config_path),
    "--read-only",
]

toolset = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command=mcp_server_command,
            args=mcp_server_args,
        )
    ),
    # Only expose the search tool. Remove this filter to also enable upsert-records, etc.
    tool_filter=["search-records"],
)


### 5b. Define the ADK Agent

The `LlmAgent` wraps the model with the MCP toolset. The `instruction` string is the system prompt -- it tells the model to always call the search tool before answering and to ground responses in the retrieved evidence.


In [ ]:
root_agent = LlmAgent(
    model=MODEL_NAME,
    name="movie_mcp_agent",
    instruction=(
        "You are a movie recommendation assistant with access to a Redis-backed movie catalog. "
        "Always call the search-records tool before answering. "
        "Ground every answer in retrieved movies from Redis, mention the titles you used, "
        "and say when the catalog does not contain a good match instead of guessing."
    ),
    tools=[toolset],
)


### 5c. Create the Session Service and Runner

ADK uses a `Runner` to execute agent turns and a `SessionService` to manage conversation state. `InMemorySessionService` is sufficient for a notebook -- in production you would use a persistent store.


In [ ]:
session_service = InMemorySessionService()
runner = Runner(
    app_name=APP_NAME,
    agent=root_agent,
    session_service=session_service,
)

print(f"Agent ready: model={MODEL_NAME}, MCP command={mcp_server_command} {' '.join(mcp_server_args)}")


## 6. Validate the MCP Connection and Define Helpers

Before running full prompts, let's verify that the MCP server starts correctly and exposes the expected tools. Then we define helper functions for interacting with the agent and extracting structured evidence from MCP tool responses.

### 6a. List Available MCP Tools

This cell starts the MCP server subprocess and prints the tools it advertises. If this fails, check that `uvx` is installed and the config path is correct.


In [ ]:
# NBVAL_SKIP
tools = await toolset.get_tools()
print(f"MCP server connected -- {len(tools)} tool(s) available:")
for tool in tools:
    print(f"  - {tool.name}: {getattr(tool, 'description', 'no description')[:80]}")


### 6b. Evidence Extraction Helpers

The MCP server returns structured JSON payloads inside the ADK event stream. These helpers extract and flatten the search results into a pandas DataFrame so we can inspect exactly which records the agent used.

- **`_model_dump`** -- converts Pydantic models to plain dicts for easier inspection.
- **`_find_search_payloads`** -- recursively walks the tool response tree looking for objects that contain `search_type` and `results` keys (the RedisVL MCP search response shape).
- **`search_evidence_dataframe`** -- combines the above to produce a flat DataFrame of matched records with their scores.


In [ ]:
def _model_dump(value):
    """Convert a Pydantic model to a JSON-serializable dict, or return the value as-is."""
    if hasattr(value, "model_dump"):
        return value.model_dump(mode="json", exclude_none=True)
    return value


def _find_search_payloads(value):
    """Recursively find RedisVL MCP search response payloads in a nested structure."""
    payloads = []
    if isinstance(value, dict):
        if "search_type" in value and "results" in value:
            payloads.append(value)
        for nested_value in value.values():
            payloads.extend(_find_search_payloads(nested_value))
    elif isinstance(value, list):
        for item in value:
            payloads.extend(_find_search_payloads(item))
    return payloads


def search_evidence_dataframe(tool_responses):
    """Extract the first search payload from tool responses and return it as a DataFrame."""
    payloads = _find_search_payloads(tool_responses)
    if not payloads:
        return None

    rows = []
    for item in payloads[0]["results"]:
        rows.append(
            {
                **item.get("record", {}),
                "score": item.get("score"),
                "score_type": item.get("score_type"),
            }
        )
    return pd.DataFrame(rows)


### 6c. The `ask_agent` Interaction Loop

This async function sends a single prompt to the agent, collects the final text response plus any MCP tool calls, and returns them in a dict. Each call creates a fresh session so prompts are independent.


In [ ]:
async def ask_agent(query: str, session_id: str | None = None):
    """Send a query to the agent and return the answer plus raw tool responses."""
    session = await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=session_id or f"session-{uuid.uuid4().hex[:8]}",
        state={},
    )

    user_message = types.Content(role="user", parts=[types.Part(text=query)])
    final_response = "No final response captured."
    tool_responses = []

    async for event in runner.run_async(
        session_id=session.id,
        user_id=session.user_id,
        new_message=user_message,
    ):
        for response in event.get_function_responses():
            tool_responses.append(_model_dump(response))

        if event.is_final_response() and event.content and event.content.parts:
            text_parts = [
                part.text for part in event.content.parts if getattr(part, "text", None)
            ]
            if text_parts:
                final_response = "\n".join(text_parts)

    return {
        "query": query,
        "answer": final_response,
        "tool_responses": tool_responses,
    }


## 7. Run a First Grounded Prompt

The next cell asks the ADK agent for a recommendation, then prints both the natural-language answer and the structured Redis evidence returned through MCP.


In [ ]:
# NBVAL_SKIP
result = await ask_agent("Recommend an action movie with gadgets and advanced technology.")
print(result["answer"])

evidence_df = search_evidence_dataframe(result["tool_responses"])
if evidence_df is not None:
    display(evidence_df[["title", "genre", "rating", "score", "score_type", "description"]])
else:
    result["tool_responses"]


## 8. Try More Prompts

These examples stay close to the movie descriptions in the sample dataset, which makes it easy to see whether the agent is grounding correctly.


In [ ]:
# NBVAL_SKIP
demo_queries = [
    "Find a funny family movie with superhero elements.",
    "Which movie best matches a revenge-driven action story?",
    "Suggest a movie about spies, technology, and a dangerous mission.",
]

for prompt in demo_queries:
    demo_result = await ask_agent(prompt)
    print(f"\nPrompt: {prompt}")
    print(demo_result["answer"])


## 9. Exercise

Try changing the agent instruction so it always returns exactly **two** movie suggestions with one sentence of justification each. Then compare how that changes the answers for a comedy-focused prompt.


In [ ]:
# Your turn:
# 1. Update `root_agent` so the answer format is stricter.
# 2. Re-run a prompt such as:
#    "Recommend a lighthearted animated movie for a family movie night."
# 3. Inspect whether the MCP evidence still matches the final answer.


## Cleanup

Close the MCP toolset (which terminates the subprocess) and drop the Redis index.


In [ ]:
# NBVAL_SKIP
await toolset.close()
index.delete(drop=True)


---

# Recap

## What We Built

1. **Redis-backed movie index** -- loaded a JSON dataset into a RedisVL search index with vector, text, tag, and numeric fields.
2. **MCP config** -- wrote a YAML file that binds the index to the RedisVL MCP server with hybrid search settings.
3. **Google ADK agent** -- connected an `LlmAgent` to the MCP server over stdio using `McpToolset`, with Vertex AI as the model backend.
4. **Evidence inspection** -- extracted structured search results from MCP tool responses so we could verify grounding.

## Why Redis for MCP?

- **Sub-millisecond retrieval** -- Redis keeps the index in memory, so even under agent-loop latency budgets, search stays fast.
- **Hybrid search** -- combining BM25 full-text scoring with vector similarity in a single query gives better relevance than either alone.
- **Unified data layer** -- the same Redis instance can serve caching, session state, and retrieval, reducing infrastructure complexity.
- **MCP compatibility** -- RedisVL's built-in MCP server means any MCP-compatible client (ADK, Claude Desktop, Cursor, etc.) can access the same index with zero custom code.

## Common Pitfalls

- **Reserved `id` field** -- RedisVL MCP uses `id` in its response envelope. If your source data has an `id` column, rename it (e.g., to `movie_id`) before loading.
- **Missing `uvx`** -- the `uvx` command comes from the [uv](https://docs.astral.sh/uv/) package manager. If it is not on your PATH, install it with `pip install uv` or see the [uv installation docs](https://docs.astral.sh/uv/getting-started/installation/).
- **Vertex AI credentials** -- if running outside Colab, make sure Application Default Credentials are set (`gcloud auth application-default login`).

## Next Steps

- Add a second notebook that enables `upsert-records` for ingestion workflows.
- Switch from the movie dataset to a domain-specific JSON corpus.
- Compare direct RedisVL querying with MCP-mediated retrieval side by side.

**Want to learn more?**
1. [RedisVL MCP documentation](https://docs.redisvl.com/en/stable/user_guide/how_to_guides/mcp.html)
2. [Google ADK MCP tools guide](https://google.github.io/adk-docs/tools/mcp-tools/)
3. [MCP protocol specification](https://modelcontextprotocol.io/)
4. [Redis AI resources repository](https://github.com/redis-developer/redis-ai-resources)
